# Registered Users: Bronze to Silver

Reads raw registered users from `projectpractice2.bronze.registered_users`, validates MAC addresses, extracts the manufacturer prefix, converts timestamps, and writes the cleaned result to `projectpractice2.silver.registered_users`.

## 1. Create silver table

In [0]:
%sql
-- Create the silver registered_users table if it does not exist yet
-- The schema is defined/overwritten by the PySpark load below (overwriteSchema = true)
CREATE TABLE IF NOT EXISTS projectpractice2.silver.registered_users;

## 2. Define bronze-to-silver transformation

Validates MAC addresses, derives `mac_prefix`, converts `registration_timestamp` from epoch seconds, casts IDs to string and adds `inserttimestamp`.

In [0]:
# Spark functions used for validation and transformation
from pyspark.sql.functions import (
    col,
    when,
    regexp_replace,
    substring,
    upper,
    from_unixtime,
    current_timestamp
)


def load_registered_users_silver():
    """
    Load registered users from bronze to silver.

    Steps:
      1. Read the bronze registered_users table
      2. Validate MAC addresses and extract the manufacturer prefix (OUI)
      3. Convert the epoch registration timestamp to TIMESTAMP
      4. Cast IDs to STRING and add an insert timestamp
      5. Overwrite the silver Delta table

    Returns the transformed DataFrame.
    """

    # ---------- Configuration ----------
    # Source and target tables
    bronze_table = "projectpractice2.bronze.registered_users"
    silver_table = "projectpractice2.silver.registered_users"

    # ---------- Extract ----------
    # Read Bronze table
    df = spark.read.table(bronze_table)

    # Valid MAC address pattern: six hex pairs separated by colons (e.g. 00:1A:2B:3C:4D:5E)
    mac_pattern = r"^([0-9A-Fa-f]{2}:){5}[0-9A-Fa-f]{2}$"

    # ---------- Transform ----------
    df = (
        df

        # Validate MAC address: flag each row as Valid / Invalid
        .withColumn(
            "validation_status",
            when(
                col("mac_address").rlike(mac_pattern),
                "Valid"
            ).otherwise("Invalid")
        )

        # Extract MAC prefix / OUI: first 6 hex chars without colons, uppercased
        # (identifies the device manufacturer)
        .withColumn(
            "mac_prefix",
            upper(
                substring(
                    regexp_replace(col("mac_address"), ":", ""),
                    1,
                    6
                )
            )
        )

        # Convert Unix epoch seconds to timestamp
        .withColumn(
            "registration_timestamp",
            from_unixtime(
                col("registration_timestamp").cast("long")
            ).cast("timestamp")
        )

        # Cast IDs to STRING (identifiers, not numeric values)
        .withColumn(
            "device_id",
            col("device_id").cast("string")
        )
        .withColumn(
            "user_id",
            col("user_id").cast("string")
        )

        # Audit column: when the row was written to silver
        .withColumn(
            "inserttimestamp",
            current_timestamp()
        )
    )

    # ---------- Load ----------
    # Write to Silver Delta table
    # overwrite + overwriteSchema replaces both data and schema on every run
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(silver_table)
    )

    print("Silver load completed successfully")

    return df

## 3. Run the silver load

In [0]:
# Execute the bronze-to-silver load for registered users
load_registered_users_silver()

## 4. Verify silver output

In [0]:
%sql
-- Preview the transformed rows in the silver table
SELECT *
FROM projectpractice2.silver.registered_users;

In [0]:
%sql
create table if not exists projectpractice2.silver.user_info_temp;

In [0]:
from pyspark.sql.functions import col, from_json, from_unixtime, to_date
from pyspark.sql.types import StructType, StructField, StringType, LongType


def load_user_info_silver():

    # ============================================================
    # 1. Source and Target Configuration
    # ============================================================

    #source_table = "projectpractice2.bronze.user_info"
    source_table = "projectpractice2.bronze.user_info_temp"
    target_table = "projectpractice2.silver.user_info_temp"
    #target_table = "user_info_temp"


    # ============================================================
    # 2. Define JSON Schema
    # ============================================================

    user_schema = StructType([
        StructField("user_id", StringType(), True),
        StructField("update_type", StringType(), True),
        StructField("timestamp", LongType(), True),
        StructField("dob", StringType(), True),
        StructField("sex", StringType(), True),
        StructField("gender", StringType(), True),
        StructField("first_name", StringType(), True),
        StructField("last_name", StringType(), True),

        StructField(
            "address",
            StructType([
                StructField("street_address", StringType(), True),
                StructField("city", StringType(), True),
                StructField("state", StringType(), True),
                StructField("zip", StringType(), True)
            ]),
            True
        )
    ])


    # ============================================================
    # 3. Read Bronze Table
    # ============================================================

    df = spark.read.table(source_table)


    # ============================================================
    # 4. Parse Raw JSON
    # ============================================================

    df_parsed = df.withColumn(
        "parsed",
        from_json(col("value"), user_schema)
    )


    # ============================================================
    # 5. Flatten and Transform Data
    # ============================================================

    df_final = df_parsed.select(

        col("parsed.user_id")
            .cast("string")
            .alias("user_id"),

        col("parsed.update_type")
            .alias("update_type"),

        from_unixtime(
            col("parsed.timestamp")
        )
        .cast("timestamp")
        .alias("converted_timestamp"),

        to_date(
            col("parsed.dob"),
            "MM/dd/yyyy"
        ).alias("dob"),

        col("parsed.sex")
            .alias("sex"),

        col("parsed.gender")
            .alias("gender"),

        col("parsed.first_name")
            .alias("first_name"),

        col("parsed.last_name")
            .alias("last_name"),

        col("parsed.address.street_address")
            .alias("street_address"),

        col("parsed.address.city")
            .alias("city"),

        col("parsed.address.state")
            .alias("state"),

        col("parsed.address.zip")
            .alias("zip")
    )


    # ============================================================
    # 6. Write to Silver Delta Table
    # ============================================================

    (
        df_final.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(target_table)
    )


    # ============================================================
    # 7. Return Processed DataFrame
    # ============================================================

    return df_final

In [0]:
%sql
create table if not exists projectpractice2.silver.user_info;

In [0]:
load_user_info_silver()
#load_user_info_silver_inc()

In [0]:
%sql
select * from projectpractice2.silver.user_info;

In [0]:
%sql
select * from projectpractice2.silver.user_info_temp;

In [0]:
from delta.tables import DeltaTable

def apply_cdc(source_table, target_table):

    # Read incoming data
    source_df = spark.read.table(source_table)

    # Reference target Delta table
    target = DeltaTable.forName(spark, target_table)

    # Apply CDC using MERGE
    (
        target.alias("t")
        .merge(
            source_df.alias("s"),
            "t.user_id = s.user_id"
        )

        # UPDATE existing records
        .whenMatchedUpdateAll()

        # INSERT new records
        .whenNotMatchedInsertAll()

        .execute()
    )


apply_cdc(
    "projectpractice2.silver.user_info_temp",
    "projectpractice2.silver.user_info"
)

In [0]:
%sql
select * from projectpractice2.silver.user_info;

In [0]:
%sql
CREATE OR REPLACE TABLE projectpractice2.silver.user_info AS
WITH ranked_data AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY user_id
            ORDER BY
                CASE
                    WHEN update_type = 'update' THEN 1
                    WHEN update_type = 'new' THEN 2
                    ELSE 3
                END,
                converted_timestamp DESC NULLS LAST
        ) AS rn
    FROM projectpractice2.silver.user_info
)

SELECT *
FROM ranked_data
WHERE rn = 1;

In [0]:
%sql
select * from projectpractice2.silver.user_info;

In [0]:
%sql
select * from projectpractice2.bronze.bpm;

In [0]:
%sql
create table if not exists projectpractice2.silver.bpm;

In [0]:
%sql
create table df5 as
select device_id.device_id, 
--time.time,
from_unixtime(time.time) as datetime,
cast(heartrate.heartrate as decimal(10,2)) as heartrate from
(select 
from_json(value, 'device_id string') as device_id,
from_json(value, 'time string') as time,
from_json(value, 'heartrate string') as heartrate
FROM projectpractice2.bronze.bpm) as A;

In [0]:
%sql
select * from df5;

In [0]:
df5 = spark.table("df5")

df5.write \
  .format("delta") \
  .mode("append") \
  .option("mergeSchema", "true") \
  .saveAsTable("projectpractice2.silver.bpm")

In [0]:
%sql
select * from projectpractice2.silver.bpm;